### Races Dimension: Silver to Gold
Join `formula1_incr.silver.races` with `formula1_incr.silver.circuits` into one table `formula1_incr.gold.dim_races`.

In [0]:
%run ../00-common/01.environment-config 

#### Setup
- `01.environment-config` → loads catalog name, silver/gold schema names
- `04.gold_helpers` → loads the `write_to_gold()` function

In [0]:
dbutils.widgets.text('p_batch_id','')
v_batch_id= dbutils.widgets.get('p_batch_id')

In [0]:
%run ../00-common/04.gold_helpers 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as f

#### Target Table

In [0]:
#silver_table = f{catalog_name}.{sil}
target_table = f'{catalog_name}.{gold_schema}.dim_races'

#### Read Silver Tables
- Read `circuits` (filtered by batch) and `races` from silver

In [0]:
circuits_df = (
    spark.table(f'{catalog_name}.{silver_schema}.circuits')
    .filter(col('batch_id') == v_batch_id)
)
races_df = (
    spark.table(f'{catalog_name}.{silver_schema}.races')
    .filter(col('batch_id') == v_batch_id)
)

#### Join
- Inner join races + circuits on `circuit_id` to get race name, date, circuit name, locality, country

In [0]:
dim_races_df = (
    races_df
    .join(circuits_df,
           races_df.circuit_id == circuits_df.circuit_id,
           'inner'
          )
    .select(races_df.season,
            races_df.round,
            races_df.race_name,
            races_df.race_date,
            circuits_df.circuit_name,
            circuits_df.locality,
            circuits_df.country
        )
)

#### Write to Gold
- If table doesn't exist, creates it fresh
- If it exists, merges using `write_to_gold()`

In [0]:
write_to_gold(
    input_df= dim_races_df,
    target_table=target_table,
    merge_condition='t.season = s.season AND t.round = s.round',
    columns_to_update=[
        'race_name',
        'race_date',
        'circuit_name',
        'locality',
        'country'
    ]
)

In [0]:
display(spark.table(target_table))